In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import sys

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))
DATA_DIR = REPO_ROOT / "DATA"
EPHYS_DIR = DATA_DIR / "ephys"

REGIONS = ["<REGION1>", "<REGION2>"]
SMOOTHING_SIGMA = 1.5

BASELINE_FILE = EPHYS_DIR / "Baseline_firing.csv"

In [ ]:
# Baseline firing rates are provided for each mouse and brain region:
# Mouse | Region | baseline (Hz)

baseline_df = pd.read_csv(BASELINE_FILE, sep=";")

baseline_lookup = (
    baseline_df
    .set_index(["Mouse", "Region"])["baseline"]
    .to_dict()
)

In [ ]:
# Load exploration-aligned activity from the JSON files generated in the previous notebook

json_files = EPHYS_DIR.rglob("*_start_interaction.json")

rows = []

for json_file in json_files:

    with json_file.open("r") as file:
        data = json.load(file)

    mouse_id = data["mouse_id"]
    region = data["region"]

    if region not in REGIONS:
        continue

    for label, label_data in data["data_by_label"].items():

        rows.append(
            {
                "mouse": mouse_id,
                "region": region,
                "label": int(float(label)),
                "psth_x": label_data["psth_x"],
                "psth_y": label_data["psth_y"],
            }
        )

df = pd.DataFrame(rows)

print(f"Loaded {len(df)} mouse x region x condition datasets.")

In [ ]:
from utils.ephys import normalize_and_smooth_psth

# Assign each mouse's region-specific baseline firing rate for normalization
df["baseline"] = [
    baseline_lookup.get((mouse, region))
    for mouse, region in zip(df["mouse"], df["region"])
]

missing_baseline = df.loc[
    df["baseline"].isna(),
    ["mouse", "region"],
].drop_duplicates()

if len(missing_baseline) > 0:
    raise ValueError(
        f"Missing baseline values for:\n{missing_baseline.to_string(index=False)}"
    )

# Normalize PSTHs to baseline firing rate and apply temporal smoothing
df["psth_norm"] = [
    normalize_and_smooth_psth(
        psth,
        baseline,
        sigma=SMOOTHING_SIGMA,
    )
    for psth, baseline in zip(
        df["psth_y"],
        df["baseline"],
    )
]

# Preserve the original PSTH for comparison with the normalized signal
df["psth_raw"] = df["psth_y"]

In [ ]:
# Generate PSTH panels for each brain region, showing
# raw activity, baseline-normalized activity, and unit-level heatmaps
# for novel and familiar exploration conditions

from utils.plot import (
    plot_raw_psth,
    plot_normalized_psth,
    plot_psth_heatmap,
)

FIGURE_DIR = REPO_ROOT / "Fig_8" / "output"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    1: "tab:blue",
    2: "tab:orange",
}

LABEL_NAMES = {
    1: "novel",
    2: "familiar",
}


fig, axes = plt.subplots(
    3,
    len(REGIONS),
    figsize=(14, 15),
    sharex="row",
)

for column, region in enumerate(REGIONS):

    region_df = df[
        df["region"] == region
    ].copy()

    region_df["peak_time_idx"] = (
        region_df["psth_norm"].apply(np.argmax)
    )

    region_df = region_df.sort_values(
        ["label", "peak_time_idx"]
    )

    long_df = region_df.explode(
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    )

    long_df[
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    ] = long_df[
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    ].apply(pd.to_numeric)

    plot_raw_psth(
        long_df,
        region,
        PALETTE,
        LABEL_NAMES,
        ax=axes[0, column],
    )

    plot_normalized_psth(
        long_df,
        PALETTE,
        LABEL_NAMES,
        ax=axes[1, column],
    )

    plot_psth_heatmap(
        region_df,
        ax=axes[2, column],
    )

    axes[0, column].set_title(region)

plt.tight_layout()
plt.show()

In [ ]:
# Save individual plot

from utils.plot import save_figure

for region in REGIONS:

    region_df = df[
        df["region"] == region
    ].copy()

    region_df["peak_time_idx"] = (
        region_df["psth_norm"].apply(np.argmax)
    )

    region_df = region_df.sort_values(
        ["label", "peak_time_idx"]
    )

    # Convert PSTH arrays to long format for plotting.
    long_df = region_df.explode(
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    )

    long_df[
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    ] = long_df[
        ["psth_x", "psth_y", "psth_norm", "psth_raw"]
    ].apply(pd.to_numeric)

    fig, ax = plt.subplots(figsize=(5, 4))

    plot_raw_psth(
        long_df,
        region,
        PALETTE,
        LABEL_NAMES,
        ax=ax,
    )

    save_figure(
        fig,
        f"{region}_raw_psth.svg",
        FIGURE_DIR,
    )

    fig, ax = plt.subplots(figsize=(5, 4))

    plot_normalized_psth(
        long_df,
        PALETTE,
        LABEL_NAMES,
        ax=ax,
    )

    save_figure(
        fig,
        f"{region}_normalized_psth.svg",
        FIGURE_DIR,
    )

    fig, ax = plt.subplots(figsize=(6, 4))

    plot_psth_heatmap(
        region_df,
        ax=ax,
    )

    save_figure(
        fig,
        f"{region}_psth_heatmap.svg",
        FIGURE_DIR,
    )

In [ ]:
# Calculate mean raw and normalized firing rates within predefined
# temporal windows around exploration onset, then export the resulting
# mouse × region × condition summary for statistical analysis

from utils.ephys import calculate_window_mean

PSTH_WINDOWS = {
    "pre": (-1, 0),
    "early": (0, 1),
    "post": (0, 3),
}

for window_name, (start_time, end_time) in PSTH_WINDOWS.items():

    df[f"raw_{window_name}"] = df.apply(
        calculate_window_mean,
        axis=1,
        start_time=start_time,
        end_time=end_time,
        column="psth_y",
    )

    df[f"norm_{window_name}"] = df.apply(
        calculate_window_mean,
        axis=1,
        start_time=start_time,
        end_time=end_time,
        column="psth_norm",
    )

    export_columns = [
    "mouse",
    "region",
    "label",
    "raw_pre",
    "norm_pre",
    "raw_early",
    "norm_early",
    "raw_post",
    "norm_post",
]

export_df = df[export_columns].copy()

OUTPUT_FILE = DATA_DIR / "psth_window_means.csv"

export_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Saved: {OUTPUT_FILE}")

In [ ]:
# Load curated unit metadata from all processed recordings and combine
# units info into a single table

from utils.ephys import load_unit_metadata

UNIT_DATA_DIR = DATA_DIR / "ephys"

unit_files = sorted(
    UNIT_DATA_DIR.rglob("*_*_units_types.csv")
)

UNIT_COLUMNS = [
    "unit_id",
    "firing_rate",
    "spike_width_ms",
    "neuron_type",
]

unit_data = [
    load_unit_metadata(file_path, UNIT_COLUMNS)
    for file_path in unit_files
]

if not unit_data:
    raise FileNotFoundError(
        f"No unit metadata files found in {UNIT_DATA_DIR}"
    )

units_df = pd.concat(
    unit_data,
    ignore_index=True,
)

print(f"Loaded {len(units_df)} units.")

print(
    units_df.groupby("region").size()
)

In [ ]:
from utils.plot import plot_unit_classification

REGION_COLORS = {
    "PFC": "red",
    "RSC": "#00BFA1",
}

OUTPUT_FILE = (
    REPO_ROOT
    / "Fig_8"
    / "output"
    / "PFC_RSC_units.svg"
)

fig, ax = plot_unit_classification(
    units_df=units_df,
    region_colors=REGION_COLORS,
    spike_width_threshold=0.3,
    firing_rate_threshold=10,
    output_file=OUTPUT_FILE,
)

plt.show()